In [9]:
import os
# Force stable download mechanisms and disable complex file-transfer sharding
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging

# Silence unnecessary download progress bars to protect VSCodium's ipykernel
logging.set_verbosity_error()

print("Loading dataset...")
data = load_dataset("knkarthick/samsum")
sample_text = data['train']['dialogue'][0]

Loading dataset...


In [7]:
if not os.path.exists(SAVE_DIR):
    print("Downloading and configuring model for the very first time (approx. 3-5 mins)...")
    model_id = "facebook/bart-large-cnn"
    
    # Clean, argument-free loading. The os.environ tag above handles the progress bars safely!
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, force_download=True)
    
    # Save it locally right now so we never have to download it again
    tokenizer.save_pretrained(SAVE_DIR)
    model.save_pretrained(SAVE_DIR)
    print(f"Model successfully saved permanently to {SAVE_DIR}!")
else:
    print("Loading model instantly from your local project storage...")
    tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
    model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR)


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]

Model successfully saved permanently to ./saved_bart_model!


In [11]:
# 2. Load instantly from your local project storage
SAVE_DIR = "./saved_bart_model"
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR)

print("Processing text inference...")
inputs = tokenizer(
    sample_text, 
    return_tensors="pt", 
    truncation=True, 
    max_length=1024
)

print("Generating final summary output...")
summary_ids = model.generate(
    inputs["input_ids"], 
    num_beams=4, 
    max_length=142, 
    early_stopping=True
)
summary = tokenizer.decode(summary_ids, skip_special_tokens=True)

print("\n" + "="*30)
print("ORIGINAL DIALOGUE:")
print("="*30)
print(sample_text)

print("\n" + "="*30)
print("SUCCESSFULLY GENERATED SUMMARY:")
print("="*30)
print(summary)

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 4634.15it/s]


Processing text inference...
Generating final summary output...

ORIGINAL DIALOGUE:
Amanda: I baked  cookies. Do you want some?
Jerry: Sure!
Amanda: I'll bring you tomorrow :-)

SUCCESSFULLY GENERATED SUMMARY:
["Amanda: I baked  cookies. Do you want some?Jerry: Sure! Amanda:  I'll bring them tomorrow :-) Jerry: I'll take them to work tomorrow. Amanda: I'm not going to work. I'm going to go home. Jerry: You're welcome."]
